# Where Philly Smells Like Coffee

Day 19 · Smell. Every coffee spot in Philadelphia mapped in OpenStreetMap, drawn as a cloud of coffee smell drifting
out over the streets. Run `python scripts\fetch_week3.py day19` first.

## 1 · Settings

In [ ]:
DAY      = 19
PROMPT   = "Smell"
SLUG     = "smell-philly-coffee"
NOTEBOOK = "day19_smell_philly_coffee.ipynb"
TITLE    = "Where Philly\nSmells Like Coffee"
CREDITS  = "Cafés, streets and water: © OpenStreetMap contributors (ODbL). City boundary: US Census Bureau."

SNIFF_M  = 250          # "you can smell it": within this many metres of a coffee spot (about a 3-minute walk)
CLOUD_M  = 220          # how far each café's smell spreads on the map (Gaussian blur, metres)
NOT_COFFEE = ("tea", "boba", "chicha", "gong cha", "mr. wish", "matcha", "bambu", "mango mango", "kung fu")
KEEP     = ("ray’s cafe", "ray's cafe")   # coffee spots with "tea" in the name
CHAINS   = ("Starbucks",)        # drawn apart from everyone else


LOOK = "espresso"        # "espresso" (dark, glowing steam), "latte" (cream, aroma contours like coffee rings), "crema" (the first draft)
ZOOM = True              # True = crop to the coffee heart of the city, with the whole city as a small inset
LOOKS = {   #  page       ink        soft       water      street     cloud faint ... strong                                 dot        chain
    "espresso": ("#1b120d", "#f3e6d3", "#a48d78", "#22303a", "#f3e6d3", ["#1b120d", "#4a2c1b", "#8a5530", "#d29a5c", "#fbe3b8"], "#fff4e2", "#7fb3a0"),
    "latte":    ("#f3eadf", "#3a2214", "#8a7464", "#dce6e6", "#3a2214", ["#f3eadf", "#e8d3b8", "#d7b089", "#b9814f", "#7a4a26"], "#3a2214", "#3f7a68"),
    "crema":    ("#f4ece1", "#2a1a12", "#8a7464", "#dfe6e4", "#2a1a12", ["#f4ece1", "#ead2b4", "#d6a878", "#a8693a", "#5b3420"], "#5b3420", "#5b6b5f"),
}
PAGE, INK, SOFT, WATER_C, STREET, CLOUD, DOT, CHAIN_C = LOOKS[LOOK]
CONTOURS = LOOK == "latte"
NEIGHBOURHOODS = {"Center City": (39.9515, -75.1650), "Fishtown": (39.9727, -75.1290), "University City": (39.9522, -75.1932),
                  "South Philly": (39.9257, -75.1700), "Germantown": (40.0390, -75.1740), "Manayunk": (40.0262, -75.2246)}

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from mapkit import poster, publish
from mapkit.paths import PROCESSED

M = "EPSG:32618"                       # metres, UTM zone 18N
cafes = gpd.read_file(PROCESSED / "philly_coffee.geojson").to_crs(M)
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'philly'").to_crs(M)
streets = gpd.read_file(PROCESSED / "philly_streets.geojson").to_crs(M)
water = gpd.read_file(PROCESSED / "philly_water.geojson").to_crs(M)
print(f"{len(cafes)} places tagged café or coffee shop · {len(streets):,} street pieces")

## 3 · Analysis, every number on the map comes from here

In [ ]:
label = (cafes["name"].fillna("") + " " + cafes["brand"].fillna("") + " " + cafes["cuisine"].fillna("")).str.lower()
tea = label.str.contains("|".join(NOT_COFFEE)) & ~label.str.contains("coffee") & ~label.str.contains("|".join(KEEP))
print("Left out as tea or dessert, not coffee:", sorted(cafes.loc[tea, "name"].fillna("(no name)").unique()))
coffee = cafes[~tea].copy()
coffee = coffee[coffee.within(city.union_all().buffer(50))]
coffee["chain"] = coffee["brand"].fillna(coffee["name"]).isin(CHAINS)
N, N_CHAIN = len(coffee), int(coffee.chain.sum())

# share of street length you can smell coffee on
sniff = coffee.buffer(SNIFF_M).union_all()
streets["m"] = streets.length
in_city = streets[streets.intersects(city.union_all())]
SMELL_SHARE = in_city.intersection(sniff).length.sum() / in_city.m.sum() * 100
SMELL_SHARE_INDIE = in_city.intersection(coffee[~coffee.chain].buffer(SNIFF_M).union_all()).length.sum() / in_city.m.sum() * 100

# busiest corner: most coffee spots within SNIFF_M of one café
xy = np.c_[coffee.geometry.x, coffee.geometry.y]
from scipy.spatial import cKDTree
counts = np.array([len(i) for i in cKDTree(xy).query_ball_point(xy, SNIFF_M)])
busiest = coffee.iloc[counts.argmax()]

print(f"{N} coffee spots: {N_CHAIN} Starbucks, {N - N_CHAIN} everyone else ({(N - N_CHAIN) / N_CHAIN:.1f} for every Starbucks)")
print(f"You can smell coffee (within {SNIFF_M} m) on {SMELL_SHARE:.0f}% of Philly's street length; "
      f"{SMELL_SHARE_INDIE:.0f}% if only the non-Starbucks count")
print(f"Busiest corner: {counts.max()} coffee spots within {SNIFF_M} m of {busiest['name']}")

## 4 · Map

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.ABRIL, poster.POPPINS, poster.POPPINS_BOLD
fig, ax = poster.page(PAGE)
ax.set_position([0.0, 0.07, 1.0, 0.76])

# the smell: every coffee spot blurred into a cloud
x0, y0, x1, y1 = city.total_bounds
res = 25
gx = np.arange(x0, x1 + res, res); gy = np.arange(y0, y1 + res, res)
H, _, _ = np.histogram2d(coffee.geometry.y, coffee.geometry.x, bins=[gy, gx])
cloud = gaussian_filter(H, CLOUD_M / res)
cloud = cloud / cloud.max()
cmap = LinearSegmentedColormap.from_list("coffee", CLOUD)
geom = city.union_all()
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MPath
clip = PathPatch(MPath.make_compound_path(*[MPath(np.asarray(p.exterior.coords)) for p in getattr(geom, "geoms", [geom])]),
                 transform=ax.transData, fc="none", ec="none")
ax.add_patch(clip)
field = np.sqrt(cloud)
if CONTOURS:                                     # aroma drawn like contour lines, or the rings a coffee cup leaves
    levels = np.linspace(0.12, 1, 9)
    cs = ax.contourf(gx[:-1] + res / 2, gy[:-1] + res / 2, field, levels=levels, cmap=cmap, zorder=1, alpha=0.9)
    cl = ax.contour(gx[:-1] + res / 2, gy[:-1] + res / 2, field, levels=levels, colors=INK, linewidths=0.5, alpha=0.55, zorder=1.5)
    for c in (cs, cl):
        c.set_clip_path(clip)
else:
    im = ax.imshow(field, origin="lower", extent=(x0, x1, y0, y1), cmap=cmap, vmin=0.04, vmax=1, zorder=1,
                   interpolation="bilinear")
    im.set_clip_path(clip)
water.plot(ax=ax, color=WATER_C, lw=0, zorder=2)
in_city.plot(ax=ax, color=STREET, lw=0.2, alpha=0.16 if LOOK == "espresso" else 0.2, zorder=3)
city.boundary.plot(ax=ax, color=INK, lw=0.8, alpha=0.7, zorder=4)
indie = coffee[~coffee.chain]; chain = coffee[coffee.chain]
dot_s = 16 if ZOOM else 9
ax.scatter(indie.geometry.x, indie.geometry.y, s=dot_s, color=DOT, edgecolor=PAGE, lw=0.5, zorder=5)
ax.scatter(chain.geometry.x, chain.geometry.y, s=dot_s * 1.8, facecolor=PAGE, edgecolor=CHAIN_C, lw=1.3, zorder=6)
if ZOOM:                                          # the coffee heart: where 80% of the coffee spots sit
    qx = np.quantile(coffee.geometry.x, [0.1, 0.9]); qy = np.quantile(coffee.geometry.y, [0.1, 0.9])
    cx_, cy_ = qx.mean(), qy.mean() + 1200
    half_h = 9000; half_w = half_h * ax.get_position().width * 10.8 / (ax.get_position().height * 13.5)
    ax.set_xlim(cx_ - half_w, cx_ + half_w); ax.set_ylim(cy_ - half_h, cy_ + half_h)
    ins = fig.add_axes([0.78, 0.70, 0.17, 0.17], facecolor="none")
    city.plot(ax=ins, color=CLOUD[1], edgecolor=INK, lw=0.6)
    ins.add_patch(plt.Rectangle((cx_ - half_w, cy_ - half_h), 2 * half_w, 2 * half_h, fill=False, ec=CLOUD[-1] if LOOK == "espresso" else INK, lw=1.2))
    ins.set_aspect("equal"); ins.set_axis_off()
else:
    ax.set_xlim(x0 - 500, x1 + 500); ax.set_ylim(y0 - 500, y1 + 500)
ax.set_aspect("equal"); ax.set_axis_off()

fig.text(0.06, 0.955, TITLE, color=INK, fontproperties=F(TITLE_F, 52), va="top", linespacing=0.82)
fig.text(0.06, 0.835, f"{N - N_CHAIN} coffee spots that aren't Starbucks. {N_CHAIN} that are.", color=SOFT,
         fontproperties=F(TEXT_F, 15), va="top")
for name, (la, lo) in NEIGHBOURHOODS.items():
    p = gpd.GeoSeries(gpd.points_from_xy([lo], [la]), crs=4326).to_crs(M).iloc[0]
    poster.halo_text(ax, p.x, p.y + 350, name, color=INK, path=TEXT_B, size=11.5 if ZOOM else 10, halo=PAGE, halo_width=3,
                     ha="center", va="bottom", zorder=8, clip_on=True)

# legend, lower right
lx, ly = 0.66, 0.20
ax.figure.add_artist(plt.matplotlib.patches.Circle((lx, ly), 0.005, transform=fig.transFigure, color=DOT))
fig.text(lx + 0.014, ly, "Coffee spot", color=INK, fontproperties=F(TEXT_F, 12), va="center")
ax.figure.add_artist(plt.matplotlib.patches.Circle((lx, ly - 0.025), 0.0055, transform=fig.transFigure, facecolor=PAGE,
                                                   edgecolor=CHAIN_C, lw=1.1))
fig.text(lx + 0.014, ly - 0.025, "Starbucks", color=INK, fontproperties=F(TEXT_F, 12), va="center")
fig.add_artist(plt.Rectangle((lx - 0.005, ly - 0.068), 0.16, 0.014, transform=fig.transFigure,
                             color="none"))
for i, c in enumerate(CLOUD[1:]):
    fig.add_artist(plt.Rectangle((lx - 0.005 + i * 0.04, ly - 0.07), 0.04, 0.014, transform=fig.transFigure, color=c, lw=0))
fig.text(lx - 0.005, ly - 0.078, "faint", color=SOFT, fontproperties=F(TEXT_F, 10), va="top")
fig.text(lx + 0.155, ly - 0.078, "strong", color=SOFT, fontproperties=F(TEXT_F, 10), va="top", ha="right")
fig.text(lx - 0.005, ly - 0.045, "COFFEE IN THE AIR", color=SOFT, fontproperties=F(TEXT_B, 9.5), va="center")

# credit, set by hand because Poppins sits taller than the helper expects
fig.text(0.06, 0.020, "Bubble tea and dessert shops left out. The smell is a drawing, not a measurement.",
         color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.038, "Cafés, streets, water © OpenStreetMap contributors · city boundary US Census",
         color=SOFT, fontproperties=F(TEXT_F, 9.5), va="bottom")
fig.text(0.06, 0.058, "Map by Gautami Kasat", color=INK, fontproperties=F(TEXT_B, 12), va="bottom")
plt.show()

## 4b · Interactive page: hover a café, click anywhere to find your nearest coffee

In [ ]:
import json

def _r(geom, nd=5):
    return json.loads(json.dumps(geom.__geo_interface__), parse_float=lambda x: round(float(x), nd))

WALK_M_PER_MIN = 80                                       # a relaxed walk, about 4.8 km/h
web_cafes = coffee.to_crs(4326).copy()
web_cafes["near"] = counts - 1                            # other coffee spots within SNIFF_M (not counting itself)
cafes_gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": _r(r.geometry), "properties": {
        "n": r["name"] if isinstance(r["name"], str) else "Unnamed café", "chain": int(r.chain), "near": int(r.near)}}
    for _, r in web_cafes.iterrows()]}
st = in_city.to_crs(4326)
st["geometry"] = in_city.geometry.simplify(4).to_crs(4326)
big_roads = ("motorway", "trunk", "primary", "secondary")
streets_gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": _r(g), "properties": {"big": int(h in big_roads)}}
    for g, h in zip(st.geometry, st.highway)]}
water_gj = {"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": _r(g), "properties": {}}
                                                       for g in water.to_crs(4326).geometry if g.geom_type in ("Polygon", "MultiPolygon")]}
city_gj = {"type": "Feature", "geometry": _r(city.to_crs(4326).union_all()), "properties": {}}
hoods_gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [lo, la]}, "properties": {"n": n}}
    for n, (la, lo) in NEIGHBOURHOODS.items()]}
cx, cy = city.to_crs(4326).union_all().centroid.coords[0]

HTML = """<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Where Philly Smells Like Coffee · Gautami Kasat</title>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Abril+Fatface&family=Poppins:wght@400;600&display=swap" rel="stylesheet">
<style>
  html,body{margin:0;height:100%;background:#1b120d;font-family:Poppins,system-ui,sans-serif;color:#f3e6d3}
  #map{position:absolute;inset:0}
  .panel{position:absolute;top:18px;left:18px;width:300px;max-width:calc(100% - 36px);background:rgba(27,18,13,.88);
         border:1px solid #4a2c1b;border-radius:12px;padding:18px 20px 14px;backdrop-filter:blur(6px)}
  h1{font-family:'Abril Fatface',serif;font-weight:400;font-size:30px;line-height:1.05;margin:0 0 12px}
  .stats{display:flex;gap:18px;margin-bottom:12px}
  .stat b{display:block;font-family:'Abril Fatface',serif;font-weight:400;font-size:34px;line-height:1}
  .stat span{font-size:12.5px;color:#a48d78}
  .radar{background:#d29a5c;color:#1b120d;border-radius:8px;padding:9px 12px;font-size:13.5px;font-weight:600;margin:4px 0 12px}
  .radar small{display:block;font-weight:400;margin-top:2px}
  label{display:flex;align-items:center;gap:8px;font-size:13.5px;cursor:pointer;margin-bottom:10px}
  .dot{width:11px;height:11px;border-radius:50%;display:inline-block}
  .credit{border-top:1px solid #4a2c1b;padding-top:8px;font-size:11px;color:#a48d78;line-height:1.45}
  .credit b{display:block;color:#f3e6d3;font-size:12.5px;font-weight:600}
  .maplibregl-popup-content{background:#f3e6d3;color:#1b120d;font-family:Poppins,sans-serif;font-size:13px;border-radius:8px;padding:9px 12px}
  .maplibregl-popup-tip{border-top-color:#f3e6d3!important;border-bottom-color:#f3e6d3!important}
  .pop b{font-size:14px}
  .hood{font-size:12px;font-weight:600;color:#f3e6d3;text-shadow:0 0 3px #1b120d,0 0 6px #1b120d;pointer-events:none;white-space:nowrap}
  @media (max-width:600px){.panel{left:10px;right:10px;width:auto;top:10px;padding:12px 14px}h1{font-size:22px}.stat b{font-size:26px}}
</style></head><body>
<div id="map"></div>
<div class="panel">
  <h1>Where Philly Smells Like Coffee</h1>
  <div class="stats">
    <div class="stat"><b>__N_INDIE__</b><span>coffee spots that<br>aren't Starbucks</span></div>
    <div class="stat"><b>__N_CHAIN__</b><span>that are</span></div>
  </div>
  <div class="radar" id="radar">☕ Click anywhere to find your nearest coffee<small>Straight-line walk, about 80 m a minute</small></div>
  <label><input type="checkbox" id="sb" checked> <span class="dot" style="border:2px solid #7fb3a0"></span> Show Starbucks</label>
  <div class="credit"><b>Map by Gautami Kasat</b>Data: © OpenStreetMap contributors (Nov 2026) · US Census 2023<br>Bubble tea and dessert shops left out.</div>
</div>
<script>
const CAFES=__CAFES__, STREETS=__STREETS__, WATER=__WATER__, CITY=__CITY__, HOODS=__HOODS__;
const map = new maplibregl.Map({container:"map", attributionControl:false, center:[__CX__, __CY__], zoom:11.2, minZoom:10, maxZoom:17,
  style:{version:8, sources:{},
         layers:[{id:"bg", type:"background", paint:{"background-color":"#1b120d"}}]}});
map.addControl(new maplibregl.NavigationControl({showCompass:false}), "bottom-right");
map.on("load", () => {
  map.addSource("city", {type:"geojson", data:CITY});
  map.addLayer({id:"land", type:"fill", source:"city", paint:{"fill-color":"#24170f"}});
  map.addSource("water", {type:"geojson", data:WATER});
  map.addLayer({id:"water", type:"fill", source:"water", paint:{"fill-color":"#22303a"}});
  map.addSource("st", {type:"geojson", data:STREETS});
  map.addLayer({id:"st-small", type:"line", source:"st", filter:["==",["get","big"],0], minzoom:12,
                paint:{"line-color":"#f3e6d3","line-opacity":0.12,"line-width":0.6}});
  map.addLayer({id:"st-big", type:"line", source:"st", filter:["==",["get","big"],1],
                paint:{"line-color":"#f3e6d3","line-opacity":0.22,"line-width":["interpolate",["linear"],["zoom"],10,0.5,15,2]}});
  map.addLayer({id:"edge", type:"line", source:"city", paint:{"line-color":"#a48d78","line-width":1,"line-opacity":0.7}});
  map.addSource("c", {type:"geojson", data:CAFES, generateId:true});
  map.addLayer({id:"smell", type:"heatmap", source:"c", paint:{
    "heatmap-radius":["interpolate",["linear"],["zoom"],10,14,13,40,16,110],
    "heatmap-intensity":["interpolate",["linear"],["zoom"],10,0.6,15,1.2],
    "heatmap-opacity":0.85,
    "heatmap-color":["interpolate",["linear"],["heatmap-density"],0,"rgba(27,18,13,0)",0.15,"#4a2c1b",0.4,"#8a5530",0.7,"#d29a5c",1,"#fbe3b8"]}});
  map.addLayer({id:"indie", type:"circle", source:"c", filter:["==",["get","chain"],0], paint:{
    "circle-radius":["interpolate",["linear"],["zoom"],10,2,15,6],"circle-color":"#fff4e2",
    "circle-stroke-color":"#1b120d","circle-stroke-width":0.8}});
  map.addLayer({id:"chain", type:"circle", source:"c", filter:["==",["get","chain"],1], paint:{
    "circle-radius":["interpolate",["linear"],["zoom"],10,3,15,7],"circle-color":"#1b120d",
    "circle-stroke-color":"#7fb3a0","circle-stroke-width":1.6}});
  HOODS.features.forEach(f => { const el = document.createElement("div"); el.className = "hood"; el.textContent = f.properties.n;
    new maplibregl.Marker({element:el, anchor:"bottom", offset:[0,-8]}).setLngLat(f.geometry.coordinates).addTo(map); });
  map.addSource("radar", {type:"geojson", data:{type:"FeatureCollection", features:[]}});
  map.addLayer({id:"radar-line", type:"line", source:"radar", paint:{"line-color":"#fbe3b8","line-width":2,"line-dasharray":[2,1.5]}});
  map.addLayer({id:"radar-you", type:"circle", source:"radar", filter:["==",["geometry-type"],"Point"],
                paint:{"circle-radius":6,"circle-color":"#d29a5c","circle-stroke-color":"#fff4e2","circle-stroke-width":2}});

  const pop = new maplibregl.Popup({closeButton:false, offset:10});
  const hover = e => { const p = e.features[0].properties; map.getCanvas().style.cursor = "pointer";
    pop.setLngLat(e.features[0].geometry.coordinates).setHTML(`<div class="pop"><b>${p.n}</b><br>
      ${p.chain ? "Starbucks" : "Independent or local chain"}<br>${p.near} other coffee spot${p.near == 1 ? "" : "s"} within a 3-minute walk</div>`).addTo(map); };
  ["indie","chain"].forEach(l => { map.on("mouseenter", l, hover); map.on("mouseleave", l, () => { pop.remove(); map.getCanvas().style.cursor = ""; }); });

  const R = 6371000, rad = d => d * Math.PI / 180;
  const metres = (a, b) => { const dl = rad(b[1]-a[1]), dg = rad(b[0]-a[0]);
    const h = Math.sin(dl/2)**2 + Math.cos(rad(a[1]))*Math.cos(rad(b[1]))*Math.sin(dg/2)**2; return 2*R*Math.asin(Math.sqrt(h)); };
  map.on("click", e => {
    if (map.queryRenderedFeatures(e.point, {layers:["indie","chain"]}).length) return;
    const you = [e.lngLat.lng, e.lngLat.lat], showSB = document.getElementById("sb").checked;
    let best = null, bd = Infinity;
    CAFES.features.forEach(f => { if (!showSB && f.properties.chain) return;
      const d = metres(you, f.geometry.coordinates); if (d < bd) { bd = d; best = f; } });
    map.getSource("radar").setData({type:"FeatureCollection", features:[
      {type:"Feature", geometry:{type:"LineString", coordinates:[you, best.geometry.coordinates]}},
      {type:"Feature", geometry:{type:"Point", coordinates:you}}]});
    const mins = Math.max(1, Math.round(bd / __WALK__));
    document.getElementById("radar").innerHTML = `☕ ${best.properties.n}<small>${mins} minute${mins == 1 ? "" : "s"} away on foot (${Math.round(bd)} m, straight line)</small>`;
  });
  document.getElementById("sb").addEventListener("change", e => map.setLayoutProperty("chain", "visibility", e.target.checked ? "visible" : "none"));
});
</script></body></html>"""

fill = {"__N_INDIE__": f"{N - N_CHAIN:,}", "__N_CHAIN__": f"{N_CHAIN:,}", "__CX__": f"{cx:.4f}", "__CY__": f"{cy:.4f}",
        "__WALK__": str(WALK_M_PER_MIN),
        "__CAFES__": json.dumps(cafes_gj, separators=(",", ":")), "__STREETS__": json.dumps(streets_gj, separators=(",", ":")),
        "__WATER__": json.dumps(water_gj, separators=(",", ":")), "__CITY__": json.dumps(city_gj, separators=(",", ":")),
        "__HOODS__": json.dumps(hoods_gj, separators=(",", ":"))}
page = HTML
for k, v in fill.items():
    page = page.replace(k, v)
print(f"page size {len(page) / 1e6:.1f} MB")

## 5 · Export: the interactive page (live map + gallery), and the still poster for LinkedIn

In [ ]:
from mapkit.paths import OUTPUT
publish.save(page, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{N - N_CHAIN} coffee spots that aren't Starbucks, {N_CHAIN} that are. Click to find your nearest coffee",
             series="city", tool="Python + MapLibre", notebook=NOTEBOOK, credits=CREDITS)
fig.savefig(OUTPUT / f"day{DAY:02d}_{SLUG}_poster.png", dpi=100, facecolor=PAGE)
print("poster     output/" + f"day{DAY:02d}_{SLUG}_poster.png")